In [ ]:
import whisper
import paddlehub as hub
import os
import csv
import time
import logging
import gc
import torch
from datetime import datetime
from pydub import AudioSegment
from opencc import OpenCC
from docx import Document
from pathlib import Path
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

whisper_model = whisper.load_model("large")
punc_model = hub.Module(name='auto_punc')

def convert_to_wav(input_file: str):
    base = Path(input_file).stem
    output_file = Path(input_file).with_name(f"{base}.wav")
    audio = AudioSegment.from_file(input_file)
    wav_audio = audio.set_channels(1).set_frame_rate(44100)
    wav_audio.export(output_file, format="wav")
    logging.info(f"轉換 WAV 成功: {output_file}")
    return str(output_file)

def my_whisper(audio_path, segment_length=300000):
    logging.info("開始進行中文語音辨識（分段+進度提示）")
    audio = AudioSegment.from_wav(audio_path)
    segments = [audio[i:i+segment_length] for i in range(0, len(audio), segment_length)]

    # 在此加入分段數量提示
    total_segments = len(segments)
    logging.info(f"音檔已分割成 {total_segments} 個分段進行處理。")

    full_text = ""
    for idx, segment in enumerate(tqdm(segments, desc="辨識進度")):
        segment.export("temp.wav", format="wav")
        result = whisper_model.transcribe("temp.wav", language='zh')
        full_text += result["text"]

        os.remove("temp.wav")
        del segment, result
        torch.cuda.empty_cache()
        gc.collect()

    logging.info("中文語音辨識完成")
    return full_text

def check_audio_file(file_name, input_file_path, is_word_correcting="N", correct_csv="ReplaceWord_v1.0.csv"):
    supported_extensions = ['aac', 'wav', 'mp3', 'm4a', 'mp4']
    for ext in supported_extensions:
        potential_file = Path(input_file_path) / f"{file_name}.{ext}"
        if potential_file.is_file():
            logging.info(f"音檔確認成功: {potential_file}")
            return str(potential_file), is_word_correcting.upper(), correct_csv
    raise FileNotFoundError("找不到音檔")

def ch_convert(transcript, method):
    return OpenCC(method).convert(transcript)

def add_punctuation(raw_script):
    def split_text(text, max_length=200):
        return [text[i:i+max_length] for i in range(0, len(text), max_length)]
    text_list = split_text(raw_script)
    processed_text = punc_model.add_puncs(text_list)
    logging.info('標點符號加入完成')
    return "".join(processed_text)

def fix_wording(fix_txt, csv_file):
    with open(csv_file, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file)
        replace_dict = {rows[0]: rows[1] for rows in reader}
    for old_word, new_word in replace_dict.items():
        fix_txt = fix_txt.replace(old_word, new_word)
    logging.info('文字更正置換完成')
    return fix_txt.replace("-", "\n")

def convert_text(input_text, file_name, output_file_path):
    output_text = Path(output_file_path) / f"{file_name}.txt"
    with open(output_text, 'w', encoding='utf-8') as file:
        file.write(input_text)
    logging.info(f"已輸出純文字檔: {output_text}")

def convert_word(input_text, file_name, output_file_path):
    output_word = Path(output_file_path) / f"{file_name}.docx"
    doc = Document()
    doc.add_paragraph(input_text)
    doc.save(output_word)
    logging.info(f"已輸出 Word 檔: {output_word}")

def main(file_list, input_file_path, is_word_correcting, correct_csv):
    for file_name in file_list:
        logging.info(f"🚩 開始處理檔案：{file_name}")
        start_time = time.time()

        input_file, is_correcting, correct_csv = check_audio_file(
            file_name, input_file_path, is_word_correcting, correct_csv
        )

        # 強制轉換成 WAV
        wav_path = convert_to_wav(input_file)
        transcript = my_whisper(wav_path)

        simplified_text = ch_convert(transcript, 'tw2s')
        processed_transcript = ch_convert(add_punctuation(simplified_text), 's2tw')

        convert_text(transcript, file_name + '_Raw', input_file_path)

        final_result = (fix_wording(processed_transcript, correct_csv)
                        if is_correcting == "Y" else processed_transcript)

        convert_text(final_result, file_name, input_file_path)
        convert_word(final_result, file_name, input_file_path)

        # 刪除中間 WAV 檔
        if os.path.exists(wav_path):
            os.remove(wav_path)
            logging.info(f"已刪除中間WAV檔案：{wav_path}")

        execution_time = (time.time() - start_time) / 60
        logging.info(f"✅ 檔案 {file_name} 處理完成，耗時：{execution_time:.2f} 分鐘\n")

    logging.info("🎉 所有檔案處理完畢！🎉")


In [ ]:


# 傳入多個檔名
file_list = [
    "a.mp3", "b.m4a", "c.aac"
]


for file_name in file_list:
    main(
        file_list=[file_name],
        input_file_path="C:/Users/interview/",
        is_word_correcting="N",
        correct_csv="ReplaceWord_v1.0.csv"
    )

    # 處理完成後清除 PyTorch 記憶體
    torch.cuda.empty_cache()
    gc.collect()
